# Solar Panel Semantic Segmentation using CNN (U-Net with ResNet Backbone)
This notebook trains a deep Convolutional Neural Network (CNN) based on the **U-Net** architecture with a pretrained **ResNet-34** backbone for solar panel segmentation.

- **Architecture**: CNN (U-Net / ResNet-34 Encoder + Decoder)
- **Input**: Satellite image patches (2021-2025)
- **Target**: Binary mask (0: Background, 1: Solar Panel)
- **Dataset Source**: Automatically cloned directly from your GitHub repository `Hinarashid220/FYP`!

## 1. Setup Environment & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install PyTorch segmentation models and helpers
!pip install -q segmentation-models-pytorch albumentations opencv-python matplotlib tqdm

## 2. Clone Dataset from GitHub
Clones `Hinarashid220/FYP` so you don't have to upload manually.

In [ ]:
import os
from pathlib import Path

# Clone repository from GitHub if not present
if not os.path.exists("FYP") and not os.path.exists("data/COCO"):
    print("Cloning dataset from GitHub repository (Hinarashid220/FYP)...")
    !git clone https://github.com/Hinarashid220/FYP.git
else:
    print("Dataset repository already available.")

# Verify dataset paths (works in Google Colab or local environment)
if os.path.exists("FYP/data/COCO"):
    DATASET_ROOT = Path("FYP/data/COCO")
elif os.path.exists("data/COCO"):
    DATASET_ROOT = Path("data/COCO")
else:
    DATASET_ROOT = Path("FYP/data/COCO")

YEARS = ["2021", "2022", "2023", "2024", "2025"]
print(f"Dataset root: {DATASET_ROOT.resolve()}")
print("Available years:", [y for y in YEARS if (DATASET_ROOT / y).exists()])

## 3. PyTorch Dataset & Mask Generator
Loads the COCO segmentation polygon coordinates and rasterizes them into binary segmentation masks for CNN training.

In [ ]:
from cnn_local import SolarCNNDataset, build_transforms, make_loader

train_dataset = SolarCNNDataset(DATASET_ROOT, YEARS, split="train", transform=build_transforms(train=True))
val_dataset = SolarCNNDataset(DATASET_ROOT, YEARS, split="valid", transform=build_transforms(train=False))
test_dataset = SolarCNNDataset(DATASET_ROOT, YEARS, split="test", transform=build_transforms(train=False))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = make_loader(train_dataset, batch_size=8, workers=0, shuffle=True, device=device)
val_loader = make_loader(val_dataset, batch_size=8, workers=0, shuffle=False, device=device)
test_loader = make_loader(test_dataset, batch_size=8, workers=0, shuffle=False, device=device)
print(f"Samples: Train={len(train_dataset)}, Valid={len(val_dataset)}, Test={len(test_dataset)}")


## 4. Visualize Dataset Sample & Ground Truth

In [ ]:
import matplotlib.pyplot as plt

img, msk = train_dataset[0]
inv_img = img.permute(1, 2, 0).cpu().numpy()
inv_img = (inv_img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
inv_img = np.clip(inv_img, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(inv_img)
axes[0].set_title("Satellite Image")
axes[0].axis("off")

axes[1].imshow(msk.squeeze().cpu().numpy(), cmap="gray")
axes[1].set_title("Solar Panel Ground Truth Mask")
axes[1].axis("off")
plt.show()

## 5. Build CNN Model (U-Net with ResNet34 Encoder)

In [ ]:
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on Device:", device)

# CNN U-Net Architecture with ResNet-34 feature extractor
model = smp.Unet(
    encoder_name="resnet34",        # Pretrained ResNet CNN encoder
    encoder_weights="imagenet",     # Transfer learning from ImageNet
    in_channels=3,                  # RGB image
    classes=1,                      # Binary segmentation: solar panel
    activation=None,                # Return logits (combined with BCEWithLogitsLoss)
).to(device)

# Loss function: Combined Dice Loss + Focal Loss (handles class imbalance)
criterion = smp.losses.DiceLoss(mode="binary", from_logits=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-6)

print("CNN Architecture:", model.__class__.__name__)
print("Backbone:", "ResNet-34")

## 6. Train CNN Model

In [ ]:
from tqdm import tqdm

def calculate_iou(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    if union == 0:
        return 1.0
    return (intersection / union).item()

EPOCHS = 35
best_val_iou = 0.0
save_dir = Path("runs/solar-seg/cnn-unet")
save_dir.mkdir(parents=True, exist_ok=True)

history = {"train_loss": [], "val_loss": [], "val_iou": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, msks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        imgs, msks = imgs.to(device), msks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, msks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)

    train_loss /= len(train_dataset)
    scheduler.step()

    # Validation loop
    model.eval()
    val_loss = 0.0
    val_ious = []
    with torch.no_grad():
        for imgs, msks in val_loader:
            imgs, msks = imgs.to(device), msks.to(device)
            preds = model(imgs)
            loss = criterion(preds, msks)
            val_loss += loss.item() * imgs.size(0)
            val_ious.append(calculate_iou(preds, msks))

    val_loss /= len(val_dataset)
    avg_val_iou = np.mean(val_ious)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(avg_val_iou)

    print(f"Epoch {epoch:02d} -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {avg_val_iou:.4f}")

    if avg_val_iou > best_val_iou:
        best_val_iou = avg_val_iou
        torch.save(model.state_dict(), save_dir / "best_cnn_unet.pth")
        print(f"  ==> Saved new best model (IoU: {best_val_iou:.4f})")

print("Training complete! Best Validation IoU:", f"{best_val_iou:.4f}")

## 7. Evaluate on Unseen Test Split

In [ ]:
# Load best weights
model.load_state_dict(torch.load(save_dir / "best_cnn_unet.pth"))
model.eval()

test_ious = []
test_f1s = []

with torch.no_grad():
    for imgs, msks in tqdm(test_loader, desc="Evaluating on Test Split"):
        imgs, msks = imgs.to(device), msks.to(device)
        preds = model(imgs)
        binary_preds = (torch.sigmoid(preds) > 0.5).float()

        for p, t in zip(binary_preds, msks):
            inter = (p * t).sum().item()
            union = (p + t).clamp(0, 1).sum().item()
            iou = (inter / union) if union > 0 else 1.0
            f1 = (2 * inter / (p.sum().item() + t.sum().item())) if (p.sum().item() + t.sum().item()) > 0 else 1.0
            test_ious.append(iou)
            test_f1s.append(f1)

print("=" * 60)
print("CNN (U-Net ResNet-34) Test Set Results:")
print(f"  Mean IoU:      {np.mean(test_ious):.4f}")
print(f"  Mean Dice/F1:  {np.mean(test_f1s):.4f}")
print("=" * 60)

## 8. Visualize Predictions & Export Results

In [ ]:
# Plot sample predictions
model.eval()
sample_imgs, sample_msks = next(iter(test_loader))
with torch.no_grad():
    sample_preds = torch.sigmoid(model(sample_imgs.to(device))) > 0.5

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i in range(3):
    # RGB Image
    img_np = sample_imgs[i].permute(1, 2, 0).cpu().numpy()
    img_np = np.clip((img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406], 0, 1)
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title("Input Image")
    axes[i, 0].axis("off")

    # Ground Truth
    axes[i, 1].imshow(sample_msks[i].squeeze().cpu().numpy(), cmap="gray")
    axes[i, 1].set_title("Ground Truth Mask")
    axes[i, 1].axis("off")

    # CNN Prediction
    axes[i, 2].imshow(sample_preds[i].squeeze().cpu().numpy(), cmap="plasma")
    axes[i, 2].set_title("CNN U-Net Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig(save_dir / "cnn_test_predictions.png", dpi=150)
plt.show()

# Zip results
!zip -r cnn_unet_results.zip runs/solar-seg/cnn-unet